# NSL-KDD scratch notebook (reconstructed)

This notebook was **rebuilt** from `docs/nsl-kdd-findings.md` after the original `scratch_randy.ipynb` was lost (it was only ever on a local Linux machine and was never committed). It follows the steps and settings the findings doc describes, with `random_state=42` throughout.

**Check against the findings doc:** the baseline (accuracy 0.78, attack recall 0.627, 259 false alarms), the threshold table, the still-missed counts at threshold 0.1, the AUCs (RF 0.958, HGB 0.959, blend 0.965) and the false-alarm-budget table all reproduce exactly. The 23-class result (accuracy 0.747, recall 0.574) matches to the rounding in the doc. The per-attack counts are close to the doc's figures but were not individually checked to the last row.

**Sections**
1. Load `KDDTrain+` / `KDDTest+`
2. Quality checks and class balance
3. Prepare features (121 columns) and the binary target
4. Random forest baseline on `KDDTest+`
5. Where the misses come from (per attack type, seen vs unseen)
6. Things that did not help (class weights)
7. Lowering the decision threshold
8. Random forest vs gradient boosting at a fixed false alarm budget

**Caveat that applies to sections 7 and 8:** thresholds and budgets are read off the test set, so the recall figures are optimistic. Fix the false alarm budget first, then report the recall at that budget.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from scipy.stats import rankdata

pd.set_option("display.width", 140)

# Find the raw data whether this notebook sits in notebooks/, notebooks/nsl-kdd/ or the repo root.
candidates = [Path("../../datasets/raw"), Path("../datasets/raw"), Path("datasets/raw")]
DATA_DIR = next(p for p in candidates if (p / "KDDTrain+.txt").exists())
print("Using data from:", DATA_DIR.resolve())

Using data from: /home/claude/geo-programme001/ai-intrusion-detection-system/datasets/raw


## 1. Load the data

The files have no header row: 41 features, then `label` (attack type as text), then `difficulty`.
Column 30 is **`srv_diff_host_rate`**. An earlier version of this notebook had it as `srv_diff_srv_rate`; that was caught by checking the names against the header of `KDDTrain+.arff`.

In [2]:
cols = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty",
]
assert len(cols) == 43 and cols[30] == "srv_diff_host_rate"

train = pd.read_csv(DATA_DIR / "KDDTrain+.txt", header=None, names=cols)
test = pd.read_csv(DATA_DIR / "KDDTest+.txt", header=None, names=cols)
print("train:", train.shape, " test:", test.shape)

train: (125973, 43)  test: (22544, 43)


## 2. Quality checks and class balance

In [3]:
print("Null values   train/test:", int(train.isna().sum().sum()), "/", int(test.isna().sum().sum()))
print("Duplicate rows train/test:", int(train.duplicated().sum()), "/", int(test.duplicated().sum()))

for name, df in [("train", train), ("test", test)]:
    normal = (df["label"] == "normal").mean()
    print(f"{name}: {normal:.1%} normal, {1 - normal:.1%} attack")

print("\nTrain labels:", train["label"].nunique(), "| rows per label (smallest 12):")
print(train["label"].value_counts().tail(12).to_string())

unseen = sorted(set(test["label"]) - set(train["label"]))
n_unseen_rows = int(test["label"].isin(unseen).sum())
n_test_attacks = int((test["label"] != "normal").sum())
print(f"\nAttack types only in test: {len(unseen)}")
print(unseen)
print(f"They cover {n_unseen_rows:,} test rows, {n_unseen_rows / n_test_attacks:.0%} of test attacks")

Null values   train/test: 0 / 0
Duplicate rows train/test: 0 / 0
train: 53.5% normal, 46.5% attack
test: 43.1% normal, 56.9% attack

Train labels: 23 | rows per label (smallest 12):
label
guess_passwd       53
buffer_overflow    30
warezmaster        20
land               18
imap               11
rootkit            10
loadmodule          9
ftp_write           8
multihop            7
phf                 4
perl                3
spy                 2

Attack types only in test: 17
['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']
They cover 3,750 test rows, 29% of test attacks


## 3. Prepare features and target

* binary target `is_attack = (label != "normal")`
* drop `difficulty` (a score attached to the dataset's labels, not available to a real detector)
* drop `num_outbound_cmds` (constant zero)
* one-hot encode `protocol_type`, `service`, `flag` → 121 columns
* reindex the test matrix to the training columns
* no scaling (tree models do not need it; add it if you try logistic regression or a neural net, fitted on train only)

In [4]:
CATEGORICAL = ["protocol_type", "service", "flag"]


def prepare(df):
    df = df.copy()
    df["is_attack"] = (df["label"] != "normal").astype(int)
    df = df.drop(columns=["num_outbound_cmds", "difficulty"], errors="ignore")
    X = pd.get_dummies(df.drop(columns=["label", "is_attack"]), columns=CATEGORICAL)
    return X, df["is_attack"], df["label"]


X, y, label_train = prepare(train)
X_test, y_test, label_test = prepare(test)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print("Feature matrix:", X.shape, "| test:", X_test.shape)
assert X.shape[1] == 121
print("Test attack share:", round(float(y_test.mean()), 3))

Feature matrix: (125973, 121) | test: (22544, 121)
Test attack share: 0.569


## 4. Baseline: random forest (100 trees)

Accuracy alone hides the problem. For a detector, **attack recall** is what matters, and it has to be read together with the number of false alarms.

In [5]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)
rf_proba = rf.predict_proba(X_test)[:, 1]
# Strict ">": with 100 trees the probabilities are multiples of 0.01, so ">" vs ">=" matters.
# scikit-learn's predict() behaves like "> 0.5" because a 50/50 tie goes to class 0.
rf_pred = (rf_proba > 0.5).astype(int)
assert (rf_pred == rf.predict(X_test)).all()

print(f"Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(classification_report(y_test, rf_pred, target_names=["Normal", "Attack"], digits=3))
tn, fp, fn, tp = confusion_matrix(y_test, rf_pred).ravel()
print(f"Confusion matrix: TN={tn}  FP(false alarms)={fp}  FN(missed attacks)={fn}  TP={tp}")

Accuracy: 0.7764
              precision    recall  f1-score   support

      Normal      0.664     0.973     0.789      9711
      Attack      0.969     0.627     0.762     12833

    accuracy                          0.776     22544
   macro avg      0.816     0.800     0.776     22544
weighted avg      0.838     0.776     0.774     22544

Confusion matrix: TN=9452  FP(false alarms)=259  FN(missed attacks)=4782  TP=8051


## 5. Where the misses come from (threshold 0.5)

In [6]:
res = pd.DataFrame({"label": label_test.values, "attack": y_test.values, "pred": rf_pred})
att = res[res["attack"] == 1].copy()
att["seen_in_training"] = ~att["label"].isin(unseen)

by_type = (
    att.groupby("label")
    .agg(total=("pred", "size"), detected=("pred", "sum"))
    .assign(recall_pct=lambda d: (100 * d["detected"] / d["total"]).round(2),
            missed=lambda d: d["total"] - d["detected"])
    .sort_values("missed", ascending=False)
)
print("Most missed attack types:")
print(by_type.head(12).to_string())

seen_unseen = att.groupby("seen_in_training").agg(total=("pred", "size"), detected=("pred", "sum"))
seen_unseen["recall_pct"] = (100 * seen_unseen["detected"] / seen_unseen["total"]).round(1)
seen_unseen["missed"] = seen_unseen["total"] - seen_unseen["detected"]
seen_unseen["share_of_all_misses_pct"] = (100 * seen_unseen["missed"] / seen_unseen["missed"].sum()).round(1)
print("\nSeen vs unseen attack types:")
print(seen_unseen.rename(index={True: "seen in training", False: "unseen in training"}).to_string())

Most missed attack types:
                 total  detected  recall_pct  missed
label                                               
guess_passwd      1231         0        0.00    1231
warezmaster        944       106       11.23     838
processtable       685         7        1.02     678
mscan              996       340       34.14     656
snmpguess          331         0        0.00     331
apache2            737       434       58.89     303
mailbomb           293         0        0.00     293
snmpgetattack      178         0        0.00     178
httptunnel         133         3        2.26     130
buffer_overflow     20         2       10.00      18
multihop            18         1        5.56      17
named               17         1        5.88      16

Seen vs unseen attack types:
                    total  detected  recall_pct  missed  share_of_all_misses_pct
seen_in_training                                                                
unseen in training   3750      1108     

## 6. Things that did not help

Class weighting. The binary classes are already near 50/50, so weights do almost nothing, and weighting a 23-class forest and collapsing back to normal vs attack is worse.

In [7]:
rf_bal = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample")
rf_bal.fit(X, y)
p = rf_bal.predict(X_test)
print(f"Binary RF, balanced_subsample:  accuracy {accuracy_score(y_test, p):.3f}, attack recall {p[y_test == 1].mean():.3f}")

rf_multi = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced_subsample")
rf_multi.fit(X, label_train)
pm = (rf_multi.predict(X_test) != "normal").astype(int)
print(f"23-class RF, collapsed:         accuracy {accuracy_score(y_test, pm):.3f}, attack recall {pm[y_test == 1].mean():.3f}")

Binary RF, balanced_subsample:  accuracy 0.778, attack recall 0.631
23-class RF, collapsed:         accuracy 0.747, attack recall 0.574


## 7. Lowering the decision threshold

The model gives many missed attacks a low but non-zero probability, so the default 0.5 cutoff throws them away. Lowering it trades extra false alarms for recall.

A connection is flagged when its attack probability is **strictly greater than** the threshold. With 100 trees the probabilities are multiples of 0.01, so `>` versus `>=` moves the numbers noticeably at low thresholds.

In [8]:
n_normal = int((y_test == 0).sum())
rows = []
for t in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05, 0.02]:
    pred = (rf_proba > t).astype(int)
    fa = int(((pred == 1) & (y_test == 0)).sum())
    rows.append({"threshold": t, "attack_recall": round(float(pred[y_test == 1].mean()), 3),
                 "false_alarms": fa, "false_alarm_pct_of_normal": round(100 * fa / n_normal, 1)})
print(pd.DataFrame(rows).to_string(index=False))

 threshold  attack_recall  false_alarms  false_alarm_pct_of_normal
      0.50          0.627           259                        2.7
      0.40          0.664           280                        2.9
      0.30          0.702           304                        3.1
      0.20          0.757           701                        7.2
      0.10          0.853           931                        9.6
      0.05          0.905           996                       10.3
      0.02          0.949          1120                       11.5


In [9]:
gp = rf_proba[(label_test == "guess_passwd").values]
print(f"guess_passwd probabilities: mean {gp.mean():.3f}, median {np.median(gp):.3f}, max {gp.max():.3f}")

still = res.assign(pred01=(rf_proba > 0.1).astype(int))
still = still[(still["attack"] == 1) & (still["pred01"] == 0)]
print(f"\nStill missed at threshold 0.1: {len(still):,}")
top = still["label"].value_counts().head(5)
print(top.to_string())
print(f"These five are {top.sum() / len(still):.0%} of the remaining misses")

guess_passwd probabilities: mean 0.067, median 0.040, max 0.320

Still missed at threshold 0.1: 1,892
label
guess_passwd     1000
mailbomb          254
warezmaster       197
snmpguess         185
snmpgetattack     178
These five are 96% of the remaining misses


## 8. Model comparison: random forest vs gradient boosting

Same `X`, `y`, `X_test`, `y_test` for both. Models are compared by recall at a fixed number of false alarms, read off each model's ROC curve.

The rank-average blend is included only to show it is not worth the complexity. It ranks scores across the test batch, so one connection's score depends on the other connections in the batch and it could not be used as-is on live traffic.

In [10]:
hgb = HistGradientBoostingClassifier(random_state=42)
hgb.fit(X, y)
hgb_proba = hgb.predict_proba(X_test)[:, 1]
blend = (rankdata(rf_proba) + rankdata(hgb_proba)) / 2

scores = {"RF": rf_proba, "HGB": hgb_proba, "Blend": blend}
print("AUC:", {k: round(roc_auc_score(y_test, v), 3) for k, v in scores.items()})

budgets = [300, 500, 750, 1000, 1500]
table = {}
for name, s in scores.items():
    fpr, tpr, _ = roc_curve(y_test, s)
    fa = fpr * n_normal
    table[name] = [round(float(tpr[np.searchsorted(fa, b, side="right") - 1]), 3) for b in budgets]
out = pd.DataFrame(table, index=pd.Index(budgets, name="false_alarms"))
out.insert(0, "pct_of_normal", [round(100 * b / n_normal, 1) for b in budgets])
print(out.to_string())

AUC: {'RF': 0.958, 'HGB': 0.959, 'Blend': 0.965}
              pct_of_normal     RF    HGB  Blend
false_alarms                                    
300                     3.1  0.696  0.759  0.724
500                     5.1  0.734  0.806  0.765
750                     7.7  0.757  0.816  0.802
1000                   10.3  0.905  0.860  0.889
1500                   15.4  0.985  0.902  0.986


## Caveats

* **Thresholds and budgets were chosen by looking at the test set**, so these recall figures are optimistic. Fix a false alarm budget in advance, then report recall at that budget.
* **Base rate:** the test set is about 43% normal traffic. A real network is overwhelmingly normal, so a 3–10% false alarm rate would produce far more alerts than this evaluation suggests.
* **Part of the gap is the data itself.** `KDDTest+` contains attack types absent from training, and attacks with very few training rows (`guess_passwd` has 53 training rows vs 1,231 in test). Tuning does not close that.
* A validation split taken from the training data would not help here because it contains no unseen attack types. This is also why every model scores about 99.9% on such a split.